# RAG01｜InfraDocs Coach：建立向量資料庫

**目標：**將 vLLM prefix caching 文件轉成可搜尋的向量資料庫，輸出 `rag-data.zip`。
本份不需要 Groq 或 Hugging Face 金鑰；第一次執行會下載 embedding 模型，CPU 即可。
執行順序：安裝 → 下載原文 → 切塊 → 向量化 → FAISS → ZIP → 檢索核對。

## 資料來源與範圍

選用 **vLLM v0.11.0 官方文件的兩個 prefix caching 章節**，用來理解模型推論基礎設施：

- [使用說明：Automatic Prefix Caching](https://github.com/vllm-project/vllm/blob/b8b302cde434df8c9289a2b465406b47ebab1c2d/docs/features/automatic_prefix_caching.md)
- [實作設計：Automatic Prefix Caching](https://github.com/vllm-project/vllm/blob/b8b302cde434df8c9289a2b465406b47ebab1c2d/docs/design/prefix_caching.md)

這是固定版本的學習資料，不宣稱涵蓋最新版。原文為英文，問題與回答使用繁體中文；專有名詞保留英文。
只處理 Markdown 中的文字與內嵌程式碼；未對圖片做 OCR，也未下載外連範例檔。
來源採 Apache-2.0 授權，ZIP 內保留原始文字與授權檔。兩個章節會依標題再分成小段。

## 1. 安裝與固定套件
若 Colab 要求重新啟動執行階段，重新啟動後再由上而下執行。

In [1]:
# Colab 安裝固定版本；本機請先安裝 requirements.txt。
import sys
import subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PACKAGES = ['sentence-transformers==5.1.1', 'transformers==4.57.1', 'huggingface-hub==0.35.3', 'faiss-cpu==1.12.0', 'aisuite[groq]==0.2.0', 'groq==0.37.1', 'gradio==6.0.1', 'numpy==2.2.6', 'pandas==2.2.3', 'matplotlib==3.10.8', 'requests==2.32.5']
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
else:
    print("本機模式：使用已安裝的環境。")

本機模式：使用已安裝的環境。


In [2]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

## 2. 固定來源與向量模型

使用 [multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small)，輸出 384 維向量。
文件加上 `passage: `，問題加上 `query: `；兩邊都做 L2 正規化。
FAISS `IndexFlatIP` 的內積因此等同餘弦相似度。模型與文件皆記錄固定版本，避免建庫與查詢不一致。

In [3]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import json
import hashlib
import re
import io
import zipfile
from pathlib import Path
import requests
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

IN_COLAB = "google.colab" in __import__("sys").modules
EMBED_MODEL = "intfloat/multilingual-e5-small"
EMBED_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
DOC_COMMIT = "b8b302cde434df8c9289a2b465406b47ebab1c2d"
DOC_VERSION = "vLLM v0.11.0"
RAW_BASE = f"https://raw.githubusercontent.com/vllm-project/vllm/{DOC_COMMIT}/"
VIEW_BASE = f"https://github.com/vllm-project/vllm/blob/{DOC_COMMIT}/"
SOURCES = [
    {"file": "automatic_prefix_caching.md", "path": "docs/features/automatic_prefix_caching.md",
     "title": "Automatic Prefix Caching — User Guide",
     "sha256": "a939df40f6cde38cc48701cfee6626dbdc73c4f1e5df46921a7b88c496e31580"},
    {"file": "prefix_caching.md", "path": "docs/design/prefix_caching.md",
     "title": "Automatic Prefix Caching — Design",
     "sha256": "0a8a9e9095055c4f8e0cdaf078e3194ab06428222d65abb9c827f6879607ce61"},
]
LICENSE_SHA = "c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4"

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def fetch_checked(url, expected_hash, limit=5_000_000):
    """下載固定版本，驗證內容；不接受登入頁面或錯誤檔案。"""
    if not url.startswith("https://"):
        raise ValueError("下載網址必須使用 HTTPS。")
    with requests.get(url, timeout=(15, 90), stream=True) as response:
        response.raise_for_status()
        parts, size = [], 0
        for part in response.iter_content(65536):
            size += len(part)
            if size > limit:
                raise ValueError("下載檔案超過預期大小。")
            parts.append(part)
    data = b"".join(parts)
    if sha256(data) != expected_hash:
        raise ValueError("檔案 SHA256 不符：請確認網址及預期雜湊，勿直接載入。")
    return data

def load_embedder():
    # 兩份 notebook 使用同一模型、版本及正規化方式；CPU 即可運作。
    # 小型資料庫以單執行緒執行，也避免本機數值函式庫的執行緒衝突。
    torch.set_num_threads(1)
    faiss.omp_set_num_threads(1)
    model = SentenceTransformer(EMBED_MODEL, revision=EMBED_REVISION,
                                device="cpu", trust_remote_code=False)
    model.max_seq_length = 512
    return model

def embed_passages(model, chunks):
    texts = [f"passage: {c['title']} / {c['section']}\n{c['text']}" for c in chunks]
    lengths = [len(model.tokenizer.encode(t)) for t in texts]
    if max(lengths, default=0) > 512:
        raise ValueError("段落超過模型長度，請縮小切塊。")
    return np.ascontiguousarray(model.encode(texts, batch_size=16,
        normalize_embeddings=True, show_progress_bar=False), dtype="float32")

def retrieve(question, model, index, chunks, k=4):
    if not isinstance(question, str) or not question.strip():
        return []
    # 即使問題是中文，E5 也要求使用英文 query: / passage: 前綴。
    vector = np.ascontiguousarray(model.encode(["query: " + question.strip()],
        normalize_embeddings=True, show_progress_bar=False), dtype="float32")
    scores, positions = index.search(vector, min(max(int(k), 1), len(chunks)))
    return [{**chunks[int(pos)], "score": float(score)}
            for score, pos in zip(scores[0], positions[0]) if pos >= 0]

def load_bundle(data, target):
    """先驗證完整 ZIP 與各檔案，再讀取 FAISS；文字中繼資料使用 JSON。"""
    allowed = {"index.faiss", "chunks.json", "manifest.json", "LICENSE-vllm.txt",
               "sources/automatic_prefix_caching.md", "sources/prefix_caching.md"}
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        entries = archive.infolist()
        if len(entries) != len(allowed) or {e.filename for e in entries} != allowed:
            raise ValueError("ZIP 的檔案清單不符預期。")
        if sum(e.file_size for e in entries) > 5_000_000:
            raise ValueError("ZIP 解壓縮大小超過限制。")
        contents = {e.filename: archive.read(e) for e in entries}
    manifest = json.loads(contents["manifest.json"])
    if (manifest.get("schema") != 1 or manifest.get("embedding_model") != EMBED_MODEL
        or manifest.get("embedding_revision") != EMBED_REVISION
        or manifest.get("normalized") is not True or manifest.get("metric") != "inner_product"):
        raise ValueError("資料庫與查詢端的 embedding 設定不一致。")
    if set(manifest["files"]) != allowed - {"manifest.json"}:
        raise ValueError("Manifest 檔案清單不完整。")
    for name, expected in manifest["files"].items():
        if sha256(contents[name]) != expected:
            raise ValueError("ZIP 內的檔案內容驗證失敗。")
    records = json.loads(contents["chunks.json"])
    required = {"id", "title", "section", "text", "url", "file"}
    if not records or any(set(c) != required or any(not isinstance(v, str) for v in c.values()) for c in records):
        raise ValueError("段落資料格式不正確。")
    if len({c["id"] for c in records}) != len(records):
        raise ValueError("段落編號重複。")
    target = Path(target)
    for name, value in contents.items():
        path = target / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(value)
    loaded = faiss.read_index(str(target / "index.faiss"))
    if (loaded.d != 384 or loaded.ntotal != len(records)
        or manifest["chunk_count"] != len(records) or manifest["dimension"] != loaded.d
        or loaded.metric_type != faiss.METRIC_INNER_PRODUCT):
        raise ValueError("向量維度、數量或距離指標不符。")
    return loaded, records, manifest

In [4]:
embedder = load_embedder()
print(f"Embedding 載入完成：{EMBED_MODEL}，維度 {embedder.get_sentence_embedding_dimension()}")

Embedding 載入完成：intfloat/multilingual-e5-small，維度 384


## 3. 依章節切塊並建立 FAISS

先按 Markdown 標題切分，再以最多 **280 tokens、重疊 48 tokens** 切塊。
每段保留來源、標題、原文與編號，方便回答時引用。程式會檢查實際 embedding 輸入沒有超過 512 tokens。

In [5]:
def sections(markdown):
    """沿章節切分；程式碼區塊內的 # 不當成標題。"""
    heading, buffer, fenced = "Overview", [], False
    for line in markdown.splitlines():
        if line.lstrip().startswith("```"):
            fenced = not fenced
        match = re.match(r"^#{1,4}\s+(.+)$", line) if not fenced else None
        bold = re.match(r"^\*\*([^*]+)\*\*\s*$", line) if not fenced else None
        if match or bold:
            if "\n".join(buffer).strip():
                yield heading, "\n".join(buffer).strip()
            heading, buffer = (match or bold).group(1), []
        else:
            # 圖片不做 OCR；保留原始文件，但不將圖片連結當成文字證據。
            if re.match(r"^!\[.*\]\(.*\)\s*$", line) or line.startswith("<gh-file:"):
                continue
            buffer.append(line)
    if "\n".join(buffer).strip():
        yield heading, "\n".join(buffer).strip()

def make_chunks(source_bytes, model, window=280, overlap=48):
    if not 0 <= overlap < window:
        raise ValueError("overlap 必須小於 window。")
    result = []
    for source in SOURCES:
        for heading, text in sections(source_bytes[source["file"]].decode("utf-8")):
            offsets = model.tokenizer(text, add_special_tokens=False,
                return_offsets_mapping=True, verbose=False)["offset_mapping"]
            for start in range(0, len(offsets), window - overlap):
                stop = min(start + window, len(offsets))
                body = text[offsets[start][0]:offsets[stop - 1][1]].strip()
                if body:
                    result.append({"id": f"C{len(result) + 1:03d}",
                        "title": source["title"], "section": heading,
                        "text": body, "url": VIEW_BASE + source["path"], "file": source["file"]})
                if stop == len(offsets):
                    break
    return result

def build_database(model, folder=Path("rag-build")):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)
    source_bytes = {s["file"]: fetch_checked(RAW_BASE + s["path"], s["sha256"]) for s in SOURCES}
    license_bytes = fetch_checked(RAW_BASE + "LICENSE", LICENSE_SHA)
    records = make_chunks(source_bytes, model)
    embeddings = embed_passages(model, records)
    vector_index = faiss.IndexFlatIP(embeddings.shape[1])
    vector_index.add(embeddings)
    faiss.write_index(vector_index, str(folder / "index.faiss"))
    contents = {"index.faiss": (folder / "index.faiss").read_bytes(),
        "chunks.json": json.dumps(records, ensure_ascii=False, indent=2).encode("utf-8"),
        "LICENSE-vllm.txt": license_bytes,
        **{"sources/" + name: data for name, data in source_bytes.items()}}
    manifest = {"schema": 1, "document_version": DOC_VERSION, "source_commit": DOC_COMMIT,
        "prepared_date": "2026-10-31", "embedding_model": EMBED_MODEL,
        "embedding_revision": EMBED_REVISION, "dimension": int(embeddings.shape[1]),
        "normalized": True, "metric": "inner_product", "chunk_count": len(records),
        "chunk_tokens": 280, "overlap_tokens": 48,
        "source_note": "Two text sections of vLLM documentation; images and external example files are not indexed.",
        "files": {name: sha256(data) for name, data in contents.items()}}
    contents["manifest.json"] = json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8")
    # 固定 ZIP metadata，讓同樣內容可以重現相同的封裝。
    zip_path = Path("rag-data.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(contents):
            entry = zipfile.ZipInfo(name, date_time=(2026, 10, 31, 0, 0, 0))
            entry.compress_type = zipfile.ZIP_DEFLATED
            entry.external_attr = 0o644 << 16
            archive.writestr(entry, contents[name])
    return vector_index, records, manifest, zip_path

In [6]:
vector_index, chunks, manifest, zip_path = build_database(embedder)
print(f"來源：{len(SOURCES)} 個章節；段落：{len(chunks)}；向量維度：{vector_index.d}")
print(f"已建立：{zip_path.name}（{zip_path.stat().st_size:,} bytes）")
print("ZIP SHA256：", sha256(zip_path.read_bytes()))
display(pd.DataFrame(chunks)[["id", "section", "file"]])

來源：2 個章節；段落：22；向量維度：384
已建立：rag-data.zip（47,655 bytes）
ZIP SHA256： 275f2a490da9df10123953df09d4cb35560bbc0b6a063570f9139475d1440864


,id,section,file
0,C001,Introduction,automatic_prefix_caching.md
1,C002,Enabling APC in vLLM,automatic_prefix_caching.md
2,C003,Example workloads,automatic_prefix_caching.md
3,C004,Limits,automatic_prefix_caching.md
4,C005,Automatic Prefix Caching,prefix_caching.md
5,C006,Automatic Prefix Caching,prefix_caching.md
6,C007,Automatic Prefix Caching,prefix_caching.md
7,C008,A hashing example with multi-modality inputs,prefix_caching.md
8,C009,A hashing example with multi-modality inputs,prefix_caching.md
9,C010,A hashing example with multi-modality inputs,prefix_caching.md


## 4. 重新載入與真實檢索檢查
這裡執行的是向量檢索，沒有呼叫 LLM；表格不是生成式回答。

In [7]:
loaded_index, loaded_chunks, loaded_manifest = load_bundle(zip_path.read_bytes(), Path("rag-reload"))
question = "APC 加速 prefill 還是 decode？哪些情況幫助不大？"
hits = retrieve(question, embedder, loaded_index, loaded_chunks, k=4)
display(pd.DataFrame([{"段落": h["id"], "章節": h["section"],
                       "相似度": round(h["score"], 4), "原文": h["text"]} for h in hits]))
assert any("decoding phase" in h["text"] for h in hits), "請檢查檢索結果是否包含 Limits。"
print("通過：重新讀入 ZIP，且前四個結果包含 prefill / decode 限制的原文。")

,段落,章節,相似度,原文
0,C004,Limits,0.8601,APC in general does not reduce the performance...
1,C003,Example workloads,0.8326,"We describe two example workloads, where APC c..."
2,C009,A hashing example with multi-modality inputs,0.8256,"```\n\nAs we can see, after the tokenization, ..."
3,C002,Enabling APC in vLLM,0.8248,Set `enable_prefix_caching=True` in vLLM engin...


通過：重新讀入 ZIP，且前四個結果包含 prefill / decode 限制的原文。


## 5. 交給 RAG02

左側檔案區的 `rag-data.zip` 就是輸出。這次已將實際建立的 ZIP 放到公開 GitHub 固定版本網址，
RAG02 可自動下載，不需助教手動上傳。若自行更換資料或重建不同的 ZIP，需重新上傳並更新 RAG02 的網址及 SHA256。
不同電腦的浮點運算可能產生不同的二進位雜湊，請使用對應該檔案的實際 SHA256。

## 來源與 AI 協作

依老師的 [RAG01](https://github.com/yenlung/AI-Demo/blob/master/【Demo06a】RAG01_打造向量資料庫.ipynb) 與
[RAG02](https://github.com/yenlung/AI-Demo/blob/master/【Demo06b】RAG02_打造_RAG_系統.ipynb) 的兩階段流程改寫。
保留 FAISS、AISuite、Groq、Gradio 的核心做法，改用免登入的 multilingual-e5-small。
資料庫使用 FAISS index + JSON，並將檔名統一為作業要求的 `rag-data.zip`。
老師範例的部分 LangChain 舊匯入路徑與 Gradio tuple 對話格式已改為本作業直接使用的 API。

本作業由 AI 協助選題、設計 Prompt、實作及檢查程式。個人理解重點：RAG 先查找資料，再將原文提供給 LLM；
它不是重新訓練模型。檢索排序與引用編號能幫助核對，但不能保證答案正確。
請附實際 Prompt／AI 回覆截圖，並以自己的話補充測試觀察。